In [2]:
%load_ext autoreload
%autoreload 2

import h5py
from tqdm import tqdm
from multiband_generate_lc import concat_light_curves
import numpy as np
import pandas as pd


In [3]:
#df_csv = pd.read_csv("data/csv/aug4_sample_chisqg10_ebv005sn3.csv", dtype={'object_id': str})

df_csv = pd.read_csv("data/aug8_stone_merged.csv", dtype={'object_id': str})

In [4]:
s82_lcs = concat_light_curves(filter_object_ids=df_csv['object_id'].values, progress_bar=True)

DEBUG concat_light_curves args: N=None, skip=None, len(filter_object_ids)=189, len(existing_object_ids)=0, save_file_path=None
Found 189 matching objects in concat_light_curves 189


Processing quasars: 100%|██████████| 189/189 [00:59<00:00,  3.19it/s]


Found 189 objects in concat_light_curves after time cut 189


Populating SDSS fields: 100%|██████████| 189/189 [00:12<00:00, 15.05it/s]


In [20]:
def save_hdf5(objs, save_file_path):
    with h5py.File(save_file_path, "w") as hdf:
        for obj in objs:
            object_id = obj["object_id"]
            group = hdf.create_group(object_id)

            # Save all attributes
            for key, value in obj.items():
                if isinstance(value, dict):
                    sub_group = group.create_group(key)
                    for sub_key, sub_value in value.items():
                        sub_group.create_dataset(sub_key, data=sub_value)
                else:
                    group.attrs[key] = value
#save_hdf5(s82_lcs, "data/csv/aug4_sample_chisqg10_ebv005sn3_lcs.h5")

def read_quasars_from_hdf5(file_path):
    quasar_list = []
    with h5py.File(file_path, "r") as hdf:
        for group_name in tqdm(hdf.keys(), desc="Reading quasars"):
            group = hdf[group_name]
            quasar = {"object_id": group_name}
            for key, value in group.attrs.items():
                quasar[key] = value
            for sub_group_name in group.keys():
                sub_group = group[sub_group_name]
                quasar[sub_group_name] = {sub_key: sub_group[sub_key][...] for sub_key in sub_group.keys()}
            quasar_list.append(quasar)
    return quasar_list

s82_lcs = read_quasars_from_hdf5("data/csv/aug4_sample_chisqg10_ebv005sn3_lcs.h5")

Reading quasars: 100%|██████████| 13193/13193 [00:18<00:00, 699.79it/s]


In [5]:
bands = ['u', 'g', 'r', 'i', 'z']

objs = []

for lc in tqdm(s82_lcs, desc="Processing light curves"):
    obj = dict(object_id=lc['object_id'],
               ra=lc['ra'], dec=lc['dec'],
               z=lc['z'],
               sdss_name=lc['sdss_name'],
               ) | {f'mags_mean_{band}': lc['mags_mean'][i] for i, band in enumerate(bands)}
    objs.append(obj)

# After the loop, convert objs to DataFrame and save to CSV
df_objs = pd.DataFrame(objs)
df_objs = df_objs.set_index('object_id').loc[df_csv['object_id']].reset_index()
df_objs.to_csv("data/csv/aug8_stone_merged_magsmean.csv", index=False)

Processing light curves: 100%|██████████| 189/189 [00:00<00:00, 613088.52it/s]
